# 01 - HITL Data Preparation

Splits the **cleaned** tweets dataset into three partitions:
- **Base** (~100 000 tweets): source for initial ground-truth labelling
- **HITL batches** (~200 000 tweets, 4 × 50 000): for iterative human review
- **Final Inference** (remainder): classified by the final model in notebook 03

Run this notebook **once** at the start of the project.

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder        = BASE_PATH / 'Raw Data/Twits/'
test_folder         = BASE_PATH / 'Raw Data/'
datasets_folder     = BASE_PATH / 'Data Sets'
cleanedds_folder    = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder     = BASE_PATH / 'Data Sets/Networks/'
literature_folder   = BASE_PATH / 'Literature/'
topic_models_folder = BASE_PATH / 'Models/Topic Modeling/'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
CLEANED_DATA_PATH = cleanedds_folder / 'cleaned_tweets.pkl'
print(f'Loading from {CLEANED_DATA_PATH}')
if CLEANED_DATA_PATH.suffix == '.pkl':
    df = pd.read_pickle(CLEANED_DATA_PATH)
else:
    df = pd.read_csv(CLEANED_DATA_PATH)
print(f'Loaded {len(df):,} tweets')

## Normalise Columns

In [ ]:
if 'public_metrics.like_count' in df.columns:
    df['likes']    = df['public_metrics.like_count']
    df['retweets'] = df['public_metrics.retweet_count']
elif 'like_count' in df.columns:
    df['likes']    = df['like_count']
    df['retweets'] = df['retweet_count']
else:
    df['likes'] = df['retweets'] = 0

if 'tweet_id' in df.columns:
    df['id'] = df['tweet_id']

keep = [c for c in ['id', 'text', 'likes', 'retweets'] if c in df.columns]
df = df[keep].copy()
df['predicted_label'] = np.nan
df['human_label']     = np.nan

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(df.head())

## Partition the Dataset

In [ ]:
BASE_SIZE = 100_000
HITL_SIZE = 200_000

if len(df) < BASE_SIZE + HITL_SIZE:
    print('Warning: dataset smaller than intended splits; adjusting.')
    BASE_SIZE = min(len(df), BASE_SIZE)
    HITL_SIZE = min(len(df) - BASE_SIZE, HITL_SIZE)

base_df      = df.iloc[:BASE_SIZE].copy()
hitl_df      = df.iloc[BASE_SIZE:BASE_SIZE + HITL_SIZE].copy()
inference_df = df.iloc[BASE_SIZE + HITL_SIZE:].copy()

print(f'Base: {len(base_df):,}  HITL: {len(hitl_df):,}  Inference: {len(inference_df):,}')

## Save Partitions

In [ ]:
hitl_folder.mkdir(parents=True, exist_ok=True)

base_df.to_pickle(hitl_folder / 'base_dataset.pkl')
inference_df.to_pickle(hitl_folder / 'inference_dataset.pkl')

BATCH_SIZE = 50_000
n_batches  = int(np.ceil(len(hitl_df) / BATCH_SIZE))
for i in range(n_batches):
    chunk = hitl_df.iloc[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
    out   = hitl_folder / f'hitl_pending_batch_{i+1:02d}.pkl'
    chunk.to_pickle(out)
    print(f'Saved {out.name} ({len(chunk):,} tweets)')

## Export Iteration-0 Review Batch

No model exists yet, so we export 10 000 random tweets as the ground-truth seed.
Label the `human_label` column and save the file before running notebook 02.

In [ ]:
seed = base_df.sample(n=min(10_000, len(base_df)), random_state=42).copy()
if 'text' in seed.columns:
    seed['text'] = seed['text'].astype(str).str.replace('\n', ' ', regex=False)

out_path = hitl_folder / 'hitl_review_batch_00.csv'
seed.to_csv(out_path, index=False)
print(f'Saved → {out_path}')